# Preprocessing

In [12]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

In [ ]:
train = pd.read_csv('/Users/adhyutabuananda/Library/Mobile Documents/com~apple~CloudDocs/BUZZMEKK/Productivity/Project/credit-scoring-pd-model/dataset/processed/train.csv')

## Fehlende Werte behandeln

In [6]:
train['monthly_income'].fillna(train['monthly_income'].median(), inplace=True)
train['num_dependents'].fillna(train['num_dependents'].median(), inplace=True)


/var/folders/y2/d_fwv8sd3pn8twwvm119x2400000gn/T/ipykernel_43719/1403166763.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['monthly_income'].fillna(train['monthly_income'].median(), inplace=True)
/var/folders/y2/d_fwv8sd3pn8twwvm119x2400000gn/T/ipykernel_43719/1403166763.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on wh

## Ausreißer kappen

In [11]:
def cap_outliers(df, col, upper_quantile = 0.9):
    upper_bound = df[col].quantile(upper_quantile)
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
    return df

for col in ['utilization', 'debt_ratio', 'monthly_income', 'past_due_30_59', 'past_due_60_89', 'past_due_90']:
    train = cap_outliers(train, col)

## Train/Validation Split

### Features & Ziel trennen

In [13]:
X = train.drop(columns=['default'])
y = train['default']

In [14]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Klassengewichtung mit SMOTE

In [15]:
smote = SMOTE(random_state=42, sampling_strategy='auto')
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'Nach SMOTE: {y_train_res.value_counts().to_dict()}')

Nach SMOTE: {0: 111979, 1: 111979}


## Datenspeichern

In [16]:
X_train_res.assign(default=y_train_res).to_csv('/Users/adhyutabuananda/Library/Mobile Documents/com~apple~CloudDocs/BUZZMEKK/Productivity/Project/credit-scoring-pd-model/dataset/processed/train_resampled.csv', index=False)
X_val.assign(default=y_val).to_csv('/Users/adhyutabuananda/Library/Mobile Documents/com~apple~CloudDocs/BUZZMEKK/Productivity/Project/credit-scoring-pd-model/dataset/processed/val.csv', index=False)